# Train your own IPT model

Put your audio in `data/raw/<your_folder>/`, with one sub-folder per playing technique.
The sub-folder names become the class names of your model.

```
data/raw/my_data_folder/
├── ordinario/
│   ├── file1.wav
│   └── file2.wav
└── staccato/
    └── file1.wav
```

Then choose how to start:

- `scratch`: train a new `base` model on your data only.
- `flute`: fine-tune the pretrained flute model on your classes.
- `guitar`: fine-tune the pretrained electric guitar model on your classes.

Fine-tuning keeps what the pretrained model learned about sound and learns your own
class names, so you can use any number of classes with any names. It usually needs less
data than training from scratch.

Edit the **Settings** cell, then run all the cells from top to bottom.

In [ ]:
# Install the requirements (only needed once per environment)
%pip install -r requirements.txt

## Settings

In [ ]:
data_folder = 'my_data_folder'  # folder inside data/raw/, one sub-folder per class
run_name = 'my_run'             # use a new name for every run
start_from = 'scratch'          # 'scratch', 'flute' or 'guitar'
epochs = 100                    # maximum number of training loops (stops earlier if it stops improving)

# Only used with start_from = 'scratch'. Fine-tuning always uses the pretrained model's values.
sr = 44100                      # sampling rate in Hz
segment_length = '14700 samps'  # classification window, in 'ms' or 'samps'; longer suits slow techniques (e.g. legato)

## Check the settings

In [ ]:
import os
import sys
import torch

# Audio settings of the pretrained models; the new data must be prepared with the same ones
PRETRAINED = {
    'flute':  dict(pretrained='pretrained/flute_pretrained.pth', model='base',
                   sr=44100, seglen='14700 samps', n_mels=384, hop_length=128, f_min=120),
    'guitar': dict(pretrained='pretrained/eguitar_help.ts', model='eguitar',
                   sr=8000, seglen='7168 samps', n_mels=128, hop_length=512, f_min=150),
}
assert start_from in ('scratch', *PRETRAINED), "start_from must be 'scratch', 'flute' or 'guitar'"

if start_from == 'scratch':
    # base expects a 384 x 115 log-mel image (384 mels, 14700 samples at hop 128):
    # the hop is scaled with the window so that any window length gives the same image
    value, unit = segment_length.split()
    assert unit in ('ms', 'samps'), "segment_length must be like '500 ms' or '14700 samps'"
    n_samples = int(value) if unit == 'samps' else int(int(value) / 1000 * sr)
    assert n_samples >= 114, 'segment_length is too short'
    cfg = dict(pretrained=None, model='base', sr=sr, seglen=segment_length, n_mels=384,
               hop_length=n_samples // 114, f_min=20)
    if cfg['hop_length'] > 512:
        print('Warning: a window this long makes the hop larger than the smallest FFT (512), '
              'part of the audio is skipped by that analysis. Use a shorter window or a lower sampling rate.')
else:
    cfg = PRETRAINED[start_from]
pretrained, model, sr, seglen = cfg['pretrained'], cfg['model'], cfg['sr'], cfg['seglen']
n_mels, hop_length, f_min = cfg['n_mels'], cfg['hop_length'], cfg['f_min']

data_path = os.path.join('data', 'raw', data_folder)
assert os.path.isdir(data_path), f'Folder not found: {data_path}'
assert not os.path.exists(os.path.join('data', 'preprocessed', run_name)), \
    f'run_name {run_name!r} was already used, choose a new one'

audio_ext = ('.wav', '.mp3', '.flac', '.aiff', '.aif')
classes = {}
for root, _, files in os.walk(data_path):
    n = sum(f.lower().endswith(audio_ext) for f in files)
    if n and root != data_path:
        classes[os.path.basename(root)] = classes.get(os.path.basename(root), 0) + n
assert len(classes) >= 2, 'Put at least two class sub-folders with audio files in your data folder'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
py = sys.executable

print(f'Start from: {start_from}  (architecture {model})')
print(f'Audio: {sr} Hz, window {seglen}, {n_mels} mels, hop {hop_length}')
print(f'Device: {device}')
print(f'{len(classes)} classes:')
for name, n in sorted(classes.items()):
    print(f'  {name}: {n} file(s)')

## Prepare the data

20 % of the files of each class are kept aside to check the model during training.

In [ ]:
!"{py}" preprocess.py --name {run_name} --train_dir {data_folder} --sampling_rate {sr} -seglen "{seglen}"

## Train

In [ ]:
if start_from == 'scratch':
    !"{py}" train.py --name {run_name} --model {model} --sampling_rate {sr} -seglen "{seglen}" --n_mels {n_mels} --hop_length {hop_length} --f_min {f_min} --device {device} --epochs {epochs} --early_stopping 10
else:
    !"{py}" finetune.py --name {run_name} --pretrained {pretrained} --model {model} --sampling_rate {sr} -seglen "{seglen}" --n_mels {n_mels} --hop_length {hop_length} --f_min {f_min} --device {device} --epochs {epochs} --early_stopping 10

## Your model

The `.ts` file is the model to load in Max with `ipt~` or `pipo.ipt`.

In [ ]:
import glob

runs = sorted(glob.glob(f'runs/{run_name}_*'), key=os.path.getmtime)
assert runs, 'No run found. Did the training finish?'
ts = glob.glob(os.path.join(runs[-1], '*.ts'))
assert ts, f'No .ts file in {runs[-1]}. Did the training finish?'
m = torch.jit.load(ts[0])
print('Model:', os.path.abspath(ts[0]))
print('Classes:', m.get_classnames())
print(f'Sampling rate: {m.get_sr()} Hz, window: {m.get_seglen()} samples')